# Backtest de Estrategias de Straddle

Este notebook prueba diferentes configuraciones de la estrategia de straddle:

1. Carga de datos y setup
2. Test de GARCH forecasts
3. Definicion de estrategias
4. Ejecucion de backtests
5. Analisis de metricas
6. Comparacion final

## 1. Setup y Carga de Datos

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import os
import sys
from datetime import datetime, timedelta

# Agregar el directorio scripts al path
scripts_path = os.path.join(os.path.dirname(os.getcwd()), 'scripts')
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

# Modulos del proyecto
from signals import (
    EntryConfig, ExitConfig,
    compute_features, compute_realized_volatility, compute_percentile_rank,
    should_enter, should_exit, get_exit_reason,
    analyze_entry_conditions, summarize_features
)
from strategy import StraddlePosition, generate_entry_dates
from backtest import run_backtest, BacktestResult
from delta_hedge import HedgeConfig
from volatility_forecast import GARCHConfig, compute_garch_forecasts, compare_rv_methods
from data_loader import get_spy_history, get_vix_history, load_treasury_curve

print("Modulos cargados correctamente")

Modulos cargados correctamente


In [2]:
# Definir periodo de analisis
start_date = '2023-01-01'
end_date = '2026-01-01'

print(f"Cargando datos desde {start_date} hasta {end_date}...")

# Cargar datos
spy_data = get_spy_history(start_date, end_date)
print(f"SPY: {len(spy_data)} registros")

vix_data = get_vix_history(start_date, end_date)
print(f"VIX: {len(vix_data)} registros")

treasury_data = load_treasury_curve(start_date, end_date)
print(f"Treasury: {len(treasury_data)} registros")

# Combinar datos
market_data = spy_data.join(vix_data, how='inner')
market_data = market_data.join(treasury_data, how='inner')

print(f"\nDatos combinados: {len(market_data)} dias de trading")
print(f"Rango: {market_data.index[0]} a {market_data.index[-1]}")

Cargando datos desde 2023-01-01 hasta 2026-01-01...
SPY: 752 registros
VIX: 752 registros
Treasury: 783 registros

Datos combinados: 752 dias de trading
Rango: 2023-01-03 a 2025-12-31


In [3]:
# Generar fechas de entrada mensuales
trading_days = market_data.index.tolist()
entry_dates = generate_entry_dates(
    start_date=trading_days[0],
    end_date=trading_days[-1],
    trading_days=trading_days,
    frequency='monthly'
)

print(f"Fechas de entrada candidatas: {len(entry_dates)}")
print(f"Primera: {entry_dates[0]}")
print(f"Ultima: {entry_dates[-1]}")

Fechas de entrada candidatas: 36
Primera: 2023-01-03
Ultima: 2025-12-01


In [4]:
# Visualizar datos historicos
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Precio SPY', 'VIX (Volatilidad Implicita)'),
    vertical_spacing=0.12,
    shared_xaxes=True
)

fig.add_trace(
    go.Scatter(x=market_data.index, y=market_data['close_spy'],
               mode='lines', name='SPY', line=dict(color='blue', width=1)),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=market_data.index, y=market_data['close_vix'] * 100,
               mode='lines', name='VIX', line=dict(color='red', width=1)),
    row=2, col=1
)

fig.add_hline(y=20, line_dash="dash", line_color="gray", opacity=0.5, row=2, col=1)

fig.update_xaxes(title_text="Fecha", row=2, col=1)
fig.update_yaxes(title_text="Precio ($)", row=1, col=1)
fig.update_yaxes(title_text="VIX (%)", row=2, col=1)

fig.update_layout(height=600, title_text="Datos Historicos: SPY y VIX", showlegend=True)
fig.show()

## 2. Test de GARCH Forecasts

Comparamos la volatilidad realizada historica con el forecast GARCH para evaluar su capacidad predictiva.

In [5]:
# Configurar modelo GARCH
garch_config = GARCHConfig(
    p=1,
    q=1,
    horizon=30,
    refit_frequency=21,
    vol_model='GARCH',
    fit_window=252
)

print("Configuracion GARCH:")
print(f"  Modelo: GARCH({garch_config.p},{garch_config.q})")
print(f"  Horizonte forecast: {garch_config.horizon} dias")
print(f"  Re-estimacion: cada {garch_config.refit_frequency} dias")
print(f"  Ventana de estimacion: {garch_config.fit_window} dias")

Configuracion GARCH:
  Modelo: GARCH(1,1)
  Horizonte forecast: 30 dias
  Re-estimacion: cada 21 dias
  Ventana de estimacion: 252 dias


In [6]:
# Calcular forecasts GARCH
print("Calculando forecasts GARCH (puede tardar unos segundos)...")
garch_forecasts = compute_garch_forecasts(market_data['close_spy'], garch_config, verbose=False)

# Estadisticas
valid_forecasts = garch_forecasts['rv_forecast'].dropna()
print(f"\nForecasts generados: {len(valid_forecasts)}")
print(f"Rango: {valid_forecasts.min()*100:.1f}% - {valid_forecasts.max()*100:.1f}%")
print(f"Media: {valid_forecasts.mean()*100:.1f}%")

Calculando forecasts GARCH (puede tardar unos segundos)...

Forecasts generados: 626
Rango: 9.4% - 34.2%
Media: 13.9%


In [7]:
# Comparar RV historica vs GARCH
comparison = compare_rv_methods(market_data['close_spy'], rv_window=20, garch_config=garch_config)
comparison = comparison.dropna()

print("Comparacion RV Historica vs GARCH Forecast:")
print(f"\nCorrelacion: {comparison['rv_historical'].corr(comparison['rv_forecast']):.3f}")
print(f"\nDiferencia (GARCH - Historica):")
print(f"  Media: {comparison['diff'].mean()*100:.2f}%")
print(f"  Std: {comparison['diff'].std()*100:.2f}%")
print(f"\nRatio (GARCH / Historica):")
print(f"  Media: {comparison['ratio'].mean():.2f}")
print(f"  Mediana: {comparison['ratio'].median():.2f}")

Comparacion RV Historica vs GARCH Forecast:

Correlacion: 0.835

Diferencia (GARCH - Historica):
  Media: 0.33%
  Std: 4.74%

Ratio (GARCH / Historica):
  Media: 1.14
  Mediana: 1.04


In [8]:
# Grafico: RV historica vs GARCH vs IV (ultimos 2 anios)
recent_start = '2023-01-01'
comparison_recent = comparison.loc[recent_start:]
iv_recent = market_data.loc[recent_start:, 'close_vix']

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=comparison_recent.index, y=comparison_recent['rv_historical'] * 100,
    mode='lines', name='RV Historica (20d)',
    line=dict(color='blue', width=1.5)
))

fig.add_trace(go.Scatter(
    x=comparison_recent.index, y=comparison_recent['rv_forecast'] * 100,
    mode='lines', name='GARCH Forecast',
    line=dict(color='green', width=1.5)
))

fig.add_trace(go.Scatter(
    x=iv_recent.index, y=iv_recent * 100,
    mode='lines', name='IV (VIX)',
    line=dict(color='red', width=1.5, dash='dash')
))

fig.update_layout(
    title='Volatilidad: RV Historica vs GARCH Forecast vs IV',
    xaxis_title='Fecha',
    yaxis_title='Volatilidad (%)',
    height=500,
    hovermode='x unified'
)

fig.show()

## 3. Definicion de Estrategias

Definimos 6 configuraciones diferentes para comparar:
- **baseline**: Sin filtros ni salidas anticipadas
- **solo_salidas**: Sin filtros de entrada, con TP/SL/TimeStop
- **filtros_rv**: Filtros basados en RV historica
- **filtros_garch**: Filtros basados en GARCH forecast
- **con_hedge**: Filtros + Delta-Hedge diario
- **agresivo_hedge**: Configuracion conservadora + GARCH + Hedge

In [9]:
backtest_configs = {
    'baseline': {
        'entry_config': EntryConfig(use_filters=False),
        'exit_config': ExitConfig(use_exits=False),
        'hedge_config': None,
        'description': 'Sin filtros, mantener hasta vencimiento'
    },
    'solo_salidas': {
        'entry_config': EntryConfig(use_filters=False),
        'exit_config': ExitConfig(
            use_exits=True,
            take_profit_pct=0.50,
            stop_loss_pct=0.50,
            time_stop_fraction=0.50,
            time_stop_min_return=0.10
        ),
        'hedge_config': None,
        'description': 'Sin filtros entrada, con TP/SL/TimeStop'
    },
    'filtros_rv': {
        'entry_config': EntryConfig(
            use_filters=True,
            ivp_threshold=30.0,
            spread_pctl_threshold=30.0,
            use_expansion_filter=True,
            use_garch_forecast=False
        ),
        'exit_config': ExitConfig(
            use_exits=True,
            take_profit_pct=0.50,
            stop_loss_pct=0.50,
            time_stop_fraction=0.50,
            time_stop_min_return=0.10
        ),
        'hedge_config': None,
        'description': 'Filtros con RV historica'
    },
    'filtros_garch': {
        'entry_config': EntryConfig(
            use_filters=True,
            ivp_threshold=30.0,
            spread_pctl_threshold=30.0,
            use_expansion_filter=True,
            use_garch_forecast=True,
            forecast_horizon=5,
            garch_refit_freq=21
        ),
        'exit_config': ExitConfig(
            use_exits=True,
            take_profit_pct=0.50,
            stop_loss_pct=0.50,
            time_stop_fraction=0.50,
            time_stop_min_return=0.10
        ),
        'hedge_config': None,
        'description': 'Filtros con GARCH forecast'
    },
    'con_hedge': {
        'entry_config': EntryConfig(
            use_filters=True,
            ivp_threshold=30.0,
            spread_pctl_threshold=30.0,
            use_expansion_filter=True
        ),
        'exit_config': ExitConfig(
            use_exits=True,
            take_profit_pct=0.50,
            stop_loss_pct=0.50,
            time_stop_fraction=0.50,
            time_stop_min_return=0.10
        ),
        'hedge_config': HedgeConfig(
            rebalance_freq='daily',
            delta_threshold=0.05,
            include_costs=True,
            cost_per_share=0.005
        ),
        'description': 'Filtros + Delta-Hedge diario'
    },
    'agresivo_hedge': {
        'entry_config': EntryConfig(
            use_filters=True,
            ivp_threshold=20.0,
            spread_pctl_threshold=20.0,
            use_expansion_filter=True,
            use_garch_forecast=True,
            forecast_horizon=5
        ),
        'exit_config': ExitConfig(
            use_exits=True,
            take_profit_pct=0.30,
            stop_loss_pct=0.30,
            time_stop_fraction=0.50,
            time_stop_min_return=0.05
        ),
        'hedge_config': HedgeConfig(
            rebalance_freq='daily',
            delta_threshold=0.05,
            include_costs=True,
            cost_per_share=0.005
        ),
        'description': 'Conservador + GARCH + Hedge'
    }
}

print("Estrategias definidas:")
for name, cfg in backtest_configs.items():
    print(f"  - {name}: {cfg['description']}")

Estrategias definidas:
  - baseline: Sin filtros, mantener hasta vencimiento
  - solo_salidas: Sin filtros entrada, con TP/SL/TimeStop
  - filtros_rv: Filtros con RV historica
  - filtros_garch: Filtros con GARCH forecast
  - con_hedge: Filtros + Delta-Hedge diario
  - agresivo_hedge: Conservador + GARCH + Hedge


## 4. Ejecucion de Backtests

In [10]:
# Ejecutar backtests
backtest_results = {}

for name, cfg in backtest_configs.items():
    print(f"Ejecutando: {name}...", end=' ')
    
    try:
        result = run_backtest(
            market_data=market_data,
            entry_dates=entry_dates,
            tenor_days=30,
            hedge_config=cfg['hedge_config'],
            entry_config=cfg['entry_config'],
            exit_config=cfg['exit_config']
        )
        backtest_results[name] = result
        print(f"OK - {result.summary.get('num_trades', 0)} trades")
    except Exception as e:
        print(f"ERROR: {e}")
        backtest_results[name] = None

print("\nBacktests completados.")

Ejecutando: baseline... OK - 36 trades
Ejecutando: solo_salidas... OK - 36 trades
Ejecutando: filtros_rv... OK - 0 trades
Ejecutando: filtros_garch... OK - 1 trades
Ejecutando: con_hedge... OK - 0 trades
Ejecutando: agresivo_hedge... OK - 0 trades

Backtests completados.


## 5. Analisis de Metricas

### 5.1 Metricas Basicas

In [11]:
# Tabla de metricas basicas
print("Comparacion de Metricas Basicas:")
print("="*110)

headers = ['Config', 'Trades', 'Win Rate', 'Total P&L', 'Avg P&L', 'Avg Win', 'Avg Loss', 'Max DD', 'PF']
print(f"{headers[0]:<16} {headers[1]:<8} {headers[2]:<10} {headers[3]:<12} {headers[4]:<10} {headers[5]:<10} {headers[6]:<10} {headers[7]:<10} {headers[8]:<8}")
print("-"*110)

for name, result in backtest_results.items():
    if result is None:
        print(f"{name:<16} ERROR")
        continue
    
    s = result.summary
    if 'error' in s:
        print(f"{name:<16} {s['error']}")
        continue
    
    print(f"{name:<16} "
          f"{s['num_trades']:<8} "
          f"{s['win_rate']*100:>6.1f}%   "
          f"${s['total_pnl_straddle']:>9.2f} "
          f"${s['avg_pnl_per_trade']:>7.2f} "
          f"${s['avg_win']:>7.2f} "
          f"${s['avg_loss']:>7.2f} "
          f"${s['max_drawdown']:>8.2f} "
          f"{s['profit_factor']:>6.2f}")

Comparacion de Metricas Basicas:
Config           Trades   Win Rate   Total P&L    Avg P&L    Avg Win    Avg Loss   Max DD     PF      
--------------------------------------------------------------------------------------------------------------
baseline         36         41.7%   $  -175.69 $  -4.88 $   6.69 $ -13.14 $-1223.47   0.36
solo_salidas     36         44.4%   $    -6.33 $  -0.18 $   3.86 $  -3.40 $ -273.15   0.91
filtros_rv       No hay trades cerrados para analizar
filtros_garch    1         100.0%   $     0.32 $   0.32 $   0.32 $   0.00 $   -1.19    inf
con_hedge        No hay trades cerrados para analizar
agresivo_hedge   No hay trades cerrados para analizar


### 5.2 Avg Holding Days

In [12]:
# Grafico de holding days por estrategia
names = []
holding_days = []

for name, result in backtest_results.items():
    if result is not None and 'error' not in result.summary:
        names.append(name)
        holding_days.append(result.summary.get('avg_holding_days', 0))

fig = go.Figure()
fig.add_trace(go.Bar(
    x=names, y=holding_days,
    marker_color='steelblue',
    text=[f"{d:.1f}" for d in holding_days],
    textposition='auto'
))

fig.update_layout(
    title='Promedio de Dias en Posicion por Estrategia',
    xaxis_title='Estrategia',
    yaxis_title='Dias Promedio',
    height=400
)

fig.show()

In [13]:
# Distribucion de holding days por estrategia
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=list(backtest_results.keys())
)

for idx, (name, result) in enumerate(backtest_results.items()):
    row = idx // 3 + 1
    col = idx % 3 + 1
    
    if result is None or 'error' in result.summary:
        continue
    
    # Calcular dias en cada trade
    days_list = []
    for t in result.trades:
        if t.status == 'closed' and t.exit_date and t.entry_date:
            days = (pd.Timestamp(t.exit_date) - pd.Timestamp(t.entry_date)).days
            days_list.append(days)
    
    if days_list:
        fig.add_trace(
            go.Histogram(x=days_list, nbinsx=15, marker_color='steelblue', showlegend=False),
            row=row, col=col
        )

fig.update_layout(height=500, title_text="Distribucion de Holding Days")
fig.show()

### 5.3 Razones de Cierre

In [14]:
def compute_exit_percentages(result):
    """Calcula % de trades por razon de cierre."""
    if 'exit_reasons' not in result.summary:
        return {'expiry': 100.0, 'take_profit': 0, 'stop_loss': 0, 'time_stop': 0, 'vol_exit': 0}
    
    total = result.summary['num_trades']
    if total == 0:
        return {'expiry': 0, 'take_profit': 0, 'stop_loss': 0, 'time_stop': 0, 'vol_exit': 0}
    
    reasons = result.summary['exit_reasons']
    return {k: v / total * 100 for k, v in reasons.items()}

In [15]:
# Tabla de razones de cierre
print("Razones de Cierre (% del total de trades):")
print("="*90)
print(f"{'Config':<16} {'Expiry':<10} {'TP':<10} {'SL':<10} {'TimeStop':<10} {'VolExit':<10}")
print("-"*90)

for name, result in backtest_results.items():
    if result is None or 'error' in result.summary:
        continue
    
    pcts = compute_exit_percentages(result)
    print(f"{name:<16} "
          f"{pcts.get('expiry', 0):>6.1f}%   "
          f"{pcts.get('take_profit', 0):>6.1f}%   "
          f"{pcts.get('stop_loss', 0):>6.1f}%   "
          f"{pcts.get('time_stop', 0):>6.1f}%   "
          f"{pcts.get('vol_exit', 0):>6.1f}%")

Razones de Cierre (% del total de trades):
Config           Expiry     TP         SL         TimeStop   VolExit   
------------------------------------------------------------------------------------------
baseline          100.0%      0.0%      0.0%      0.0%      0.0%
solo_salidas        0.0%      5.6%      0.0%     63.9%     30.6%
filtros_garch       0.0%      0.0%      0.0%    100.0%      0.0%


In [16]:
# Grafico de barras apiladas de razones de cierre
exit_data = {'Config': [], 'Expiry': [], 'Take Profit': [], 'Stop Loss': [], 'Time Stop': [], 'Vol Exit': []}

for name, result in backtest_results.items():
    if result is None or 'error' in result.summary:
        continue
    
    pcts = compute_exit_percentages(result)
    exit_data['Config'].append(name)
    exit_data['Expiry'].append(pcts.get('expiry', 0))
    exit_data['Take Profit'].append(pcts.get('take_profit', 0))
    exit_data['Stop Loss'].append(pcts.get('stop_loss', 0))
    exit_data['Time Stop'].append(pcts.get('time_stop', 0))
    exit_data['Vol Exit'].append(pcts.get('vol_exit', 0))

fig = go.Figure()

colors = {'Expiry': 'gray', 'Take Profit': 'green', 'Stop Loss': 'red', 'Time Stop': 'orange', 'Vol Exit': 'purple'}

for reason in ['Expiry', 'Take Profit', 'Stop Loss', 'Time Stop', 'Vol Exit']:
    fig.add_trace(go.Bar(
        name=reason,
        x=exit_data['Config'],
        y=exit_data[reason],
        marker_color=colors[reason]
    ))

fig.update_layout(
    barmode='stack',
    title='Distribucion de Razones de Cierre por Estrategia',
    xaxis_title='Estrategia',
    yaxis_title='Porcentaje (%)',
    height=500
)

fig.show()

### 5.4 Distribucion de IVP y SpreadVar en Entradas

In [17]:
def compute_entry_distributions(result):
    """Extrae IVP y SpreadVar de trades en fecha de entrada."""
    if result.features is None:
        return {'ivp': [], 'spread': []}
    
    features = result.features
    ivp_at_entry = []
    spread_at_entry = []
    
    for trade in result.trades:
        entry_date = trade.entry_date
        if entry_date in features.index:
            row = features.loc[entry_date]
            if not pd.isna(row['iv_pctl']):
                ivp_at_entry.append(row['iv_pctl'])
            if not pd.isna(row['spread_var_pctl']):
                spread_at_entry.append(row['spread_var_pctl'])
    
    return {'ivp': ivp_at_entry, 'spread': spread_at_entry}

In [18]:
# Histogramas de IVP y SpreadVar en entradas
strategies_with_filters = ['filtros_rv', 'filtros_garch', 'con_hedge', 'agresivo_hedge']

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'IVP en Entradas (filtros_rv vs filtros_garch)',
        'SpreadVar Pctl en Entradas (filtros_rv vs filtros_garch)',
        'IVP en Entradas (con_hedge vs agresivo_hedge)',
        'SpreadVar Pctl en Entradas (con_hedge vs agresivo_hedge)'
    ]
)

# filtros_rv vs filtros_garch - IVP
for name, color in [('filtros_rv', 'blue'), ('filtros_garch', 'green')]:
    if name in backtest_results and backtest_results[name] is not None:
        dist = compute_entry_distributions(backtest_results[name])
        if dist['ivp']:
            fig.add_trace(
                go.Histogram(x=dist['ivp'], name=name, marker_color=color, opacity=0.6, nbinsx=10),
                row=1, col=1
            )

# filtros_rv vs filtros_garch - SpreadVar
for name, color in [('filtros_rv', 'blue'), ('filtros_garch', 'green')]:
    if name in backtest_results and backtest_results[name] is not None:
        dist = compute_entry_distributions(backtest_results[name])
        if dist['spread']:
            fig.add_trace(
                go.Histogram(x=dist['spread'], name=name, marker_color=color, opacity=0.6, nbinsx=10, showlegend=False),
                row=1, col=2
            )

# con_hedge vs agresivo_hedge - IVP
for name, color in [('con_hedge', 'orange'), ('agresivo_hedge', 'red')]:
    if name in backtest_results and backtest_results[name] is not None:
        dist = compute_entry_distributions(backtest_results[name])
        if dist['ivp']:
            fig.add_trace(
                go.Histogram(x=dist['ivp'], name=name, marker_color=color, opacity=0.6, nbinsx=10),
                row=2, col=1
            )

# con_hedge vs agresivo_hedge - SpreadVar
for name, color in [('con_hedge', 'orange'), ('agresivo_hedge', 'red')]:
    if name in backtest_results and backtest_results[name] is not None:
        dist = compute_entry_distributions(backtest_results[name])
        if dist['spread']:
            fig.add_trace(
                go.Histogram(x=dist['spread'], name=name, marker_color=color, opacity=0.6, nbinsx=10, showlegend=False),
                row=2, col=2
            )

fig.update_layout(height=600, barmode='overlay', title_text='Distribucion de Condiciones de Entrada')
fig.update_xaxes(title_text='IVP', row=1, col=1)
fig.update_xaxes(title_text='SpreadVar Pctl', row=1, col=2)
fig.update_xaxes(title_text='IVP', row=2, col=1)
fig.update_xaxes(title_text='SpreadVar Pctl', row=2, col=2)

fig.show()

### 5.4.1 Comparacion RV vs IV y P&L de Estrategias con Filtro de Spread

Este grafico muestra la relacion entre volatilidad realizada e implicita, junto con el P&L de las estrategias que usan `spread_pctl_threshold` como criterio de entrada.

- **Eje principal (izq)**: IV (VIX) y RV historica (20d)
- **Eje secundario (der)**: P&L acumulado de estrategias con filtro de spread
- **Puntos**: Trades individuales (verde = ganancia, rojo = perdida)

In [19]:
# Estrategias que usan spread_pctl_threshold
spread_strategies = ['filtros_rv', 'filtros_garch', 'con_hedge', 'agresivo_hedge']

def extract_spread_strategy_trades(backtest_results, backtest_configs, features):
    """
    Extrae informacion de trades de estrategias con spread_pctl_threshold.
    
    Returns
    -------
    list of dict
        Lista con info de cada trade: fecha entrada/salida, P&L, spread, estrategia
    """
    trades_data = []
    
    for name in spread_strategies:
        if name not in backtest_results or backtest_results[name] is None:
            continue
        
        result = backtest_results[name]
        if 'error' in result.summary:
            continue
        
        threshold = backtest_configs[name]['entry_config'].spread_pctl_threshold
        
        for trade in result.trades:
            if trade.status != 'closed':
                continue
            
            entry_date = pd.Timestamp(trade.entry_date)
            exit_date = pd.Timestamp(trade.exit_date) if trade.exit_date else None
            pnl = trade.pnl_total if hasattr(trade, 'pnl_total') else (trade.exit_price - trade.entry_price if trade.exit_price else 0)
            
            # Obtener IV y RV en fecha de entrada
            if entry_date in features.index:
                row = features.loc[entry_date]
                iv_entry = row['iv'] * 100 if not pd.isna(row['iv']) else np.nan
                rv_entry = row['rv_long'] * 100 if not pd.isna(row['rv_long']) else np.nan
                spread_pctl = row['spread_var_pctl'] if not pd.isna(row['spread_var_pctl']) else np.nan
            else:
                iv_entry, rv_entry, spread_pctl = np.nan, np.nan, np.nan
            
            trades_data.append({
                'strategy': name,
                'threshold': threshold,
                'entry_date': entry_date,
                'exit_date': exit_date,
                'pnl': pnl,
                'iv_entry': iv_entry,
                'rv_entry': rv_entry,
                'spread_pctl': spread_pctl,
                'spread_diff': iv_entry - rv_entry if not pd.isna(iv_entry) and not pd.isna(rv_entry) else np.nan
            })
    
    return trades_data

# Extraer trades
spread_trades = extract_spread_strategy_trades(backtest_results, backtest_configs, comparison)
print(f"Trades de estrategias con spread_pctl_threshold: {len(spread_trades)}")
for t in spread_trades:
    print(f"  {t['strategy']}: {t['entry_date'].strftime('%Y-%m-%d')} -> P&L=${t['pnl']:.2f}, SpreadPctl={t['spread_pctl']:.1f}%")

Trades de estrategias con spread_pctl_threshold: 1
  filtros_garch: 2024-06-03 -> P&L=$0.32, SpreadPctl=nan%


In [20]:
# Grafico: RV vs IV con P&L de estrategias con spread_pctl_threshold
from plotly.subplots import make_subplots

# Preparar datos de volatilidad
vol_data = comparison.dropna()

# Calcular P&L acumulado combinado de estrategias con spread
combined_pnl = pd.Series(dtype=float)
for name in spread_strategies:
    if name in backtest_results and backtest_results[name] is not None:
        if 'error' not in backtest_results[name].summary:
            pnl_series = backtest_results[name].cumulative_pnl_total
            if combined_pnl.empty:
                combined_pnl = pnl_series.copy()
            else:
                combined_pnl = combined_pnl.add(pnl_series, fill_value=0)

# Crear figura con eje secundario
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Eje principal: IV y RV
fig.add_trace(
    go.Scatter(
        x=vol_data.index,
        y=vol_data['rv_historical'] * 100,
        mode='lines',
        name='RV Historica (20d)',
        line=dict(color='blue', width=1.5),
        hovertemplate='%{x}<br>RV: %{y:.1f}%<extra></extra>'
    ),
    secondary_y=False
)

fig.add_trace(
    go.Scatter(
        x=market_data.index,
        y=market_data['close_vix'] * 100,
        mode='lines',
        name='IV (VIX)',
        line=dict(color='red', width=1.5, dash='dash'),
        hovertemplate='%{x}<br>IV: %{y:.1f}%<extra></extra>'
    ),
    secondary_y=False
)

# Eje secundario: P&L acumulado (area sombreada)
if not combined_pnl.empty:
    fig.add_trace(
        go.Scatter(
            x=combined_pnl.index,
            y=combined_pnl.values,
            mode='lines',
            name='P&L Acumulado (Spread Strategies)',
            line=dict(color='green', width=2, shape='hv'),
            fill='tozeroy',
            fillcolor='rgba(0, 128, 0, 0.1)',
            hovertemplate='%{x}<br>P&L: $%{y:.2f}<extra></extra>'
        ),
        secondary_y=True
    )

# Scatter de trades individuales
if spread_trades:
    trade_dates = [t['entry_date'] for t in spread_trades]
    trade_pnls = [t['pnl'] for t in spread_trades]
    trade_colors = ['green' if p > 0 else 'red' for p in trade_pnls]
    trade_sizes = [max(8, min(25, abs(p) / 2 + 8)) for p in trade_pnls]
    
    # Posicion Y: usar IV en fecha de entrada para colocar el punto
    trade_y = [t['iv_entry'] for t in spread_trades]
    
    # Texto hover personalizado
    hover_texts = []
    for t in spread_trades:
        text = (
            f"<b>{t['strategy']}</b><br>"
            f"Entrada: {t['entry_date'].strftime('%Y-%m-%d')}<br>"
            f"Salida: {t['exit_date'].strftime('%Y-%m-%d') if t['exit_date'] else 'N/A'}<br>"
            f"P&L: ${t['pnl']:.2f}<br>"
            f"IV: {t['iv_entry']:.1f}% | RV: {t['rv_entry']:.1f}%<br>"
            f"Spread (IV-RV): {t['spread_diff']:.1f}%<br>"
            f"Spread Pctl: {t['spread_pctl']:.1f}%"
        )
        hover_texts.append(text)
    
    fig.add_trace(
        go.Scatter(
            x=trade_dates,
            y=trade_y,
            mode='markers',
            name='Trades (Spread Strategies)',
            marker=dict(
                color=trade_colors,
                size=trade_sizes,
                line=dict(color='white', width=1),
                symbol='circle'
            ),
            hovertemplate='%{hovertext}<extra></extra>',
            hovertext=hover_texts
        ),
        secondary_y=False
    )

# Configurar ejes
fig.update_xaxes(title_text='Fecha')
fig.update_yaxes(title_text='Volatilidad (%)', secondary_y=False)
fig.update_yaxes(title_text='P&L Acumulado ($)', secondary_y=True)

# Layout
fig.update_layout(
    title='RV vs IV con P&L de Estrategias con Filtro de Spread (spread_pctl_threshold)',
    height=600,
    hovermode='x unified',
    legend=dict(
        yanchor='top',
        y=0.99,
        xanchor='left',
        x=0.01,
        bgcolor='rgba(255,255,255,0.8)'
    )
)

# Linea de referencia en P&L = 0
fig.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.5, secondary_y=True)

fig.show()

### 5.5 Metricas de Delta-Hedge

In [21]:
def compute_hedge_metrics(result):
    """Calcula metricas detalladas del hedge."""
    if not result.hedge_trades:
        return None
    
    turnover = sum(abs(t.shares_traded) for t in result.hedge_trades)
    total_costs = sum(t.trade_cost for t in result.hedge_trades)
    num_rebal = len(result.hedge_trades)
    avg_cost = total_costs / num_rebal if num_rebal > 0 else 0
    
    return {
        'turnover': turnover,
        'total_costs': total_costs,
        'num_rebalances': num_rebal,
        'avg_cost_per_rebal': avg_cost,
        'pnl_straddle': result.summary['total_pnl_straddle'],
        'pnl_hedge': result.summary.get('total_pnl_hedge', 0),
        'pnl_total': result.summary['total_pnl']
    }

In [22]:
# Tabla de metricas de hedge
print("Metricas de Delta-Hedge:")
print("="*100)
print(f"{'Config':<16} {'Rebalances':<12} {'Turnover':<14} {'Costes Tot':<12} {'Coste/Rebal':<12} {'P&L Straddle':<14} {'P&L Hedge':<12}")
print("-"*100)

for name in ['con_hedge', 'agresivo_hedge']:
    if name not in backtest_results or backtest_results[name] is None:
        continue
    
    metrics = compute_hedge_metrics(backtest_results[name])
    if metrics is None:
        continue
    
    print(f"{name:<16} "
          f"{metrics['num_rebalances']:<12} "
          f"{metrics['turnover']:>12.0f} "
          f"${metrics['total_costs']:>9.2f} "
          f"${metrics['avg_cost_per_rebal']:>9.2f} "
          f"${metrics['pnl_straddle']:>12.2f} "
          f"${metrics['pnl_hedge']:>10.2f}")

Metricas de Delta-Hedge:
Config           Rebalances   Turnover       Costes Tot   Coste/Rebal  P&L Straddle   P&L Hedge   
----------------------------------------------------------------------------------------------------


In [23]:
# Grafico de contribucion al P&L para estrategias con hedge
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['con_hedge', 'agresivo_hedge']
)

for idx, name in enumerate(['con_hedge', 'agresivo_hedge']):
    if name not in backtest_results or backtest_results[name] is None:
        continue
    
    result = backtest_results[name]
    col = idx + 1
    
    # P&L Straddle
    fig.add_trace(
        go.Scatter(
            x=result.cumulative_pnl_straddle.index,
            y=result.cumulative_pnl_straddle.values,
            mode='lines', name='Straddle',
            line=dict(color='blue', width=1.5),
            showlegend=(idx == 0)
        ),
        row=1, col=col
    )
    
    # P&L Hedge
    fig.add_trace(
        go.Scatter(
            x=result.cumulative_pnl_hedge.index,
            y=result.cumulative_pnl_hedge.values,
            mode='lines', name='Hedge',
            line=dict(color='green', width=1.5),
            showlegend=(idx == 0)
        ),
        row=1, col=col
    )
    
    # Costes acumulados (si disponibles)
    if result.hedge_trades:
        costs_by_date = {}
        cumulative_cost = 0
        for t in result.hedge_trades:
            cumulative_cost += t.trade_cost
            costs_by_date[t.date] = cumulative_cost
        
        if costs_by_date:
            costs_series = pd.Series(costs_by_date)
            fig.add_trace(
                go.Scatter(
                    x=costs_series.index,
                    y=-costs_series.values,
                    mode='lines', name='Costes (neg)',
                    line=dict(color='red', width=1, dash='dash'),
                    showlegend=(idx == 0)
                ),
                row=1, col=col
            )
    
    # P&L Total
    fig.add_trace(
        go.Scatter(
            x=result.cumulative_pnl_total.index,
            y=result.cumulative_pnl_total.values,
            mode='lines', name='Total',
            line=dict(color='black', width=2),
            showlegend=(idx == 0)
        ),
        row=1, col=col
    )

fig.update_layout(
    height=400,
    title_text='Contribucion al P&L: Straddle vs Hedge vs Costes'
)
fig.update_yaxes(title_text='P&L ($)')

fig.show()

## 6. Comparacion Final

In [24]:
# P&L Acumulado de todas las estrategias
fig = go.Figure()

colors_map = {
    'baseline': 'gray',
    'solo_salidas': 'blue',
    'filtros_rv': 'green',
    'filtros_garch': 'orange',
    'con_hedge': 'purple',
    'agresivo_hedge': 'red'
}

for name, result in backtest_results.items():
    if result is not None and 'error' not in result.summary:
        fig.add_trace(go.Scatter(
            x=result.cumulative_pnl_total.index,
            y=result.cumulative_pnl_total.values,
            mode='lines',
            name=f"{name} (${result.summary['total_pnl']:.0f})",
            line=dict(color=colors_map.get(name, 'black'), width=2)
        ))

fig.add_hline(y=0, line_color="black", line_width=1, line_dash="dash")

fig.update_layout(
    title='Comparacion de Estrategias: P&L Acumulado',
    xaxis_title='Fecha',
    yaxis_title='P&L Acumulado ($)',
    height=600,
    hovermode='x unified',
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()

In [25]:
# Tabla comparativa completa
comparison_data = []

for name, result in backtest_results.items():
    if result is None or 'error' in result.summary:
        continue
    
    s = result.summary
    exit_pcts = compute_exit_percentages(result)
    hedge_metrics = compute_hedge_metrics(result)
    
    row = {
        'Config': name,
        'Trades': s['num_trades'],
        'Win Rate': f"{s['win_rate']*100:.1f}%",
        'Total P&L': f"${s['total_pnl']:.2f}",
        'Max DD': f"${s['max_drawdown']:.2f}",
        'Profit Factor': f"{s['profit_factor']:.2f}",
        'Avg Days': f"{s.get('avg_holding_days', 0):.1f}",
        '% TP': f"{exit_pcts.get('take_profit', 0):.1f}",
        '% SL': f"{exit_pcts.get('stop_loss', 0):.1f}",
        '% TimeStop': f"{exit_pcts.get('time_stop', 0):.1f}",
        '% VolExit': f"{exit_pcts.get('vol_exit', 0):.1f}",
        '% Expiry': f"{exit_pcts.get('expiry', 0):.1f}",
        'Hedge Turnover': f"{hedge_metrics['turnover']:.0f}" if hedge_metrics else '-',
        'Hedge Costs': f"${hedge_metrics['total_costs']:.2f}" if hedge_metrics else '-'
    }
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)
print("Tabla Comparativa Completa:")
print("="*150)
print(comparison_df.to_string(index=False))

Tabla Comparativa Completa:
       Config  Trades Win Rate Total P&L    Max DD Profit Factor Avg Days % TP % SL % TimeStop % VolExit % Expiry Hedge Turnover Hedge Costs
     baseline      36    41.7% $-1058.88 $-1223.47          0.36     30.5  0.0  0.0        0.0       0.0    100.0              -           -
 solo_salidas      36    44.4%  $-195.45  $-273.15          0.91     12.2  5.6  0.0       63.9      30.6      0.0              -           -
filtros_garch       1   100.0%    $24.60    $-1.19           inf     21.0  0.0  0.0      100.0       0.0      0.0              -           -


In [26]:
# Visualizacion de metricas clave
names = []
metrics = {'num_trades': [], 'win_rate': [], 'total_pnl': [], 
           'max_dd': [], 'profit_factor': [], 'avg_holding': []}

for name, result in backtest_results.items():
    if result is not None and 'error' not in result.summary:
        names.append(name)
        s = result.summary
        metrics['num_trades'].append(s['num_trades'])
        metrics['win_rate'].append(s['win_rate'] * 100)
        metrics['total_pnl'].append(s['total_pnl'])
        metrics['max_dd'].append(s['max_drawdown'])
        metrics['profit_factor'].append(min(s['profit_factor'], 5))
        metrics['avg_holding'].append(s.get('avg_holding_days', 0))

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=(
        'Numero de Trades',
        'Win Rate',
        'P&L Total',
        'Max Drawdown',
        'Profit Factor',
        'Avg Holding Days'
    ),
    vertical_spacing=0.15,
    horizontal_spacing=0.1
)

fig.add_trace(go.Bar(x=names, y=metrics['num_trades'], marker_color='steelblue', showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=names, y=metrics['win_rate'], marker_color='green', showlegend=False), row=1, col=2)

pnl_colors = ['green' if v > 0 else 'red' for v in metrics['total_pnl']]
fig.add_trace(go.Bar(x=names, y=metrics['total_pnl'], marker_color=pnl_colors, showlegend=False), row=1, col=3)

fig.add_trace(go.Bar(x=names, y=metrics['max_dd'], marker_color='red', showlegend=False), row=2, col=1)
fig.add_trace(go.Bar(x=names, y=metrics['profit_factor'], marker_color='purple', showlegend=False), row=2, col=2)
fig.add_trace(go.Bar(x=names, y=metrics['avg_holding'], marker_color='orange', showlegend=False), row=2, col=3)

fig.add_hline(y=50, line_dash="dash", line_color="red", opacity=0.5, row=1, col=2)
fig.add_hline(y=0, line_color="black", line_width=1, row=1, col=3)
fig.add_hline(y=1, line_dash="dash", line_color="red", opacity=0.5, row=2, col=2)

fig.update_layout(height=600, title_text="Metricas Clave por Estrategia", showlegend=False)
fig.show()

In [27]:
# Ranking por Profit Factor
ranking_data = []

for name, result in backtest_results.items():
    if result is not None and 'error' not in result.summary:
        s = result.summary
        ranking_data.append({
            'Estrategia': name,
            'Profit Factor': s['profit_factor'],
            'Win Rate': s['win_rate'] * 100,
            'Total P&L': s['total_pnl'],
            'Trades': s['num_trades']
        })

ranking_df = pd.DataFrame(ranking_data)
ranking_df = ranking_df.sort_values('Profit Factor', ascending=False).reset_index(drop=True)
ranking_df.index = ranking_df.index + 1

print("Ranking por Profit Factor:")
print("="*70)
print(ranking_df.to_string())

Ranking por Profit Factor:
      Estrategia  Profit Factor    Win Rate    Total P&L  Trades
1  filtros_garch            inf  100.000000    24.600969       1
2   solo_salidas       0.906985   44.444444  -195.449331      36
3       baseline       0.363453   41.666667 -1058.881337      36


### Conclusiones

Los resultados del backtest permiten comparar el impacto de:

1. **Filtros de entrada**: Las estrategias con filtros (IVP, SpreadVar, expansion) reducen el numero de trades pero pueden mejorar el profit factor al seleccionar mejores puntos de entrada.

2. **GARCH vs RV Historica**: El uso de GARCH forecast puede capturar mejor la dinamica de volatilidad que la RV historica, lo que se refleja en las distribuciones de IVP y SpreadVar en los puntos de entrada.

3. **Salidas anticipadas**: Las reglas de TP/SL/TimeStop reducen el holding period promedio y pueden limitar perdidas, aunque tambien pueden cortar ganancias.

4. **Delta-Hedge**: El hedge reduce la exposicion direccional pero introduce costes de rebalanceo. El impacto neto depende de la volatilidad del mercado durante el periodo.